In [23]:
import time

import sys
import pandas as pd
sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait 
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC

options = webdriver.ChromeOptions() # Usamos chrome, se podria usar otro.
options.add_argument('--headless') # Chromium sin interfaz grafica
options.add_argument('--no-sandbox') # Seguridad
options.add_argument('--disable-dev-shm-usage') # configuracion de linux
options.add_argument('--user-agent=""Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/74.0.3729.157 Safari/537.36""') # user agent

# driver = webdriver.Chrome()

# CAPEC

def mapping(links_capec):
    '''
    Buscar si hay patrones de ataques relacionados
    con ID CAPEC
    
    Arg: lista
    return: lista
    
    '''
    
    links_attack = []
    for i in links_capec:
        url_capec = i
        try:
            print(url_capec)
            time.sleep(2)
            wd.get(url_capec)
            
            try:
                # busco si hay patrones de ataques relacionados
                attack_patterns = wd.find_element(By.ID, "Taxonomy_Mappings")
                # En caso de que exista busco los map_attack
                href_attack = attack_patterns.find_elements(By.TAG_NAME,"a")
                
                time.sleep(2)
                
                for i in href_attack:
                    print(f'mirando urls internas{i.get_attribute("href")}')

                time.sleep(5)
                for i in href_attack:
                    
                    link_aux = i.get_attribute("href")
                    print(link_aux)
                    # valido url
                    if link_aux.startswith('https://capec.mitre.org'):
                        # link_aux = [link_aux]
                        
                        link_recurs = mapping([link_aux])
                        print(f'{link_recurs} viene de la recurividad')
                        links_attack.append(link_recurs)
                        return links_attack
                        
                    elif link_aux.startswith('https://attack.mitre.org/'):
                        links_attack.append(link_aux)
                        print(f"guarodo el link: {link_aux}")
                    else:
                        # print("No guardo nada")
                        pass

            except EC.NoSuchElementException:
                print(f"{url_capec}: No contiene una correlación con mitre")
        
        except EC.WebDriverException:
            print(f"{url_capec}: No es una url")
    return links_attack

In [14]:
cve = input("Ingresar un CVE:")
# CVE-2025-20122
# CVE-2024-40591
# CVE-2024-24914
# CVE-2025-20210
# CVE-2025-64155
# CVE-2025-59503
# CVE-2026-20045



Ingresar un CVE: CVE-2026-20045


In [15]:
# Configuramos el web driver
wd = webdriver.Chrome()

# Navegamos la pág NIST con el CVE
url = f"https://nvd.nist.gov/vuln/detail/{cve}"
wd.get(url)

# text_box = wd.find_element(By.ID, "vulnDescriptionTitle")
text_href = wd.find_element(By.XPATH, '//*[@id="vulnTechnicalDetailsDiv"]/table/tbody/tr/td[1]/a')
print(text_href.text)
print(text_href.get_attribute("href"))

CWE-94
http://cwe.mitre.org/data/definitions/94.html


In [16]:
# CWE Common Weakness Enumeration
url_cwe = text_href.get_attribute("href")
wd.get(url_cwe)


try:
    #busco si hay patrones de ataques relacionados
    attack_patterns = wd.find_element(By.ID, "Related_Attack_Patterns")
    
    # En caso de que exista, busco los capec
    href_capec = attack_patterns.find_elements(By.TAG_NAME,"a")
    
    # Guardo los links en una lista
    links_capec = []
    for i in href_capec:
        links_capec.append(i.get_attribute("href"))

except EC.NoSuchElementException:
    print("No contiene patrones de ataque")
    links_capec = []
    wd.quit()
        
print(links_capec)

["javascript:toggleblocksOC('94_Related_Attack_Patterns');", 'http://capec.mitre.org/data/definitions/242.html', 'http://capec.mitre.org/data/definitions/35.html', 'http://capec.mitre.org/data/definitions/77.html']


In [24]:
links = links_capec
lista_mitre = mapping(links)

lista_mitre

javascript:toggleblocksOC('94_Related_Attack_Patterns');
javascript:toggleblocksOC('94_Related_Attack_Patterns');: No es una url
http://capec.mitre.org/data/definitions/242.html
mirando urls internasjavascript:toggleblocksOC('242_Taxonomy Mappings');
mirando urls internashttps://owasp.org/www-community/attacks/Code_Injection
javascript:toggleblocksOC('242_Taxonomy Mappings');
https://owasp.org/www-community/attacks/Code_Injection
http://capec.mitre.org/data/definitions/35.html
mirando urls internasjavascript:toggleblocksOC('35_Taxonomy Mappings');
mirando urls internashttps://capec.mitre.org/data/definitions/636.html
mirando urls internashttps://attack.mitre.org/wiki/Technique/T1027/006
mirando urls internashttps://attack.mitre.org/wiki/Technique/T1027/009
mirando urls internashttps://attack.mitre.org/wiki/Technique/T1564/009
javascript:toggleblocksOC('35_Taxonomy Mappings');
https://capec.mitre.org/data/definitions/636.html
https://capec.mitre.org/data/definitions/636.html
mirando url

[[['https://attack.mitre.org/wiki/Technique/T1036/003']]]

In [9]:
with open("map_mitre_detections.txt",'w') as f:
    for i in lista_mitre:
        f.write(f'{i}\n')
    

In [10]:
wd.quit()

In [11]:
!pip freeze > requierement.txt